# Retail Sales Forecasting & Business Intelligence — Milestone 1
**Participant:** Participant_M1_03  
**Project:** Retail Sales Forecasting & Business Intelligence  
**Milestone:** 1  
**Dataset package:** `Retail_(TS).zip`

## Objective
Forecast daily sales for each store-item combination and support inventory/replenishment decisions.

## Milestone scope
parse and sort dates; inspect missing dates and sales values; EDA by date/store/item; create time features; naive and seasonal-naive baselines; Linear Regression; chronological validation.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## 1. Load, inspect, and validate time series
A missing store-item-date record is not automatically interpreted as zero sales; the meaning must be verified.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

df = pd.read_csv(Path("data") / "retail_data.csv", parse_dates=["date"])
df = df.sort_values(["store", "item", "date"])
print("shape:", df.shape)
print(df.dtypes)
print("date range:", df.date.min(), "to", df.date.max())
print("stores/items:", df.store.nunique(), df.item.nunique())
print("duplicates:", df.duplicated(["date", "store", "item"]).sum())
print(df.isna().sum())
display(df.head())

## 2. EDA and baseline forecasting
Use chronological splitting. Features for a date must only use observations available before that date.

In [ ]:
import matplotlib.pyplot as plt
sample = df[(df["store"] == df["store"].iloc[0]) & (df["item"] == df["item"].iloc[0])]
sample.set_index("date")["sales"].plot(title="Example store-item daily sales", figsize=(10, 4))
plt.show()
display(df.groupby(df["date"].dt.month)["sales"].agg(["mean", "median", "sum"]))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df["dayofweek"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["lag_1"] = df.groupby(["store", "item"])["sales"].shift(1)
df["lag_7"] = df.groupby(["store", "item"])["sales"].shift(7)
model_df = df.dropna(subset=["lag_1", "lag_7"]).copy()
cutoff = model_df["date"].quantile(0.8)
train = model_df[model_df["date"] <= cutoff]
test = model_df[model_df["date"] > cutoff]
features = ["store", "item", "dayofweek", "month", "lag_1", "lag_7"]
lr = LinearRegression().fit(train[features], train["sales"])
pred = lr.predict(test[features])
print({"MAE": mean_absolute_error(test["sales"], pred),
       "RMSE": mean_squared_error(test["sales"], pred) ** 0.5})

## 3. Caveats
- The simple quantile cutoff is only a demonstration; use an explicit time cutoff for the actual forecast horizon.
- Verify whether missing dates mean zero demand or missing records.
- Add a naive and seasonal-naive baseline before claiming the regression model is useful.

## Additional review notes
The baseline workflow is included, but further work is needed to document the effect of each cleaning choice, compare against a simple baseline, and explain errors by relevant groups.